# Retail Superstore ETL Pipeline

## Dataset
Sample Superstore Sales Dataset, source: kaggle

Location:
`D:\Retail sales projects\Project project`

## Objective

Transform the raw 21-column retail transaction dataset into a PostgreSQL-ready star schema.

Workflow:

Raw CSV
→ Data Quality Check
→ Cleaning
→ Dimension Tables
→ Fact Table
→ Export for SQL

Final tables:

- dim_customer
- dim_product
- dim_location
- dim_date
- fact_sales


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

print("Libraries loaded")


Libraries loaded


## 1. Load Raw Dataset

The dataset contains:

- Order information
- Customer information
- Product information
- Location information
- Sales metrics


In [3]:
# Importing

file_path = r"D:\Retail sales projects\Project project\Superstore.csv"

df = pd.read_csv(
    file_path,
    encoding="latin1"
)

print("Shape:", df.shape)

df.head()

Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
df.info()

df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

## 2. Data Cleaning

Cleaning steps:

- Remove duplicates
- Convert dates
- Fix column names
- Handle missing postal codes


In [5]:
df = df.drop_duplicates()

df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

df["Postal Code"] = df["Postal Code"].fillna(0).astype(int)

df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 3. Create Customer Dimension

In [6]:
dim_customer = (
    df[
        [
            "customer_id",
            "customer_name",
            "segment"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_customer.head()

,customer_id,customer_name,segment
0,CG-12520,Claire Gute,Consumer
1,DV-13045,Darrin Van Huff,Corporate
2,SO-20335,Sean O'Donnell,Consumer
3,BH-11710,Brosina Hoffman,Consumer
4,AA-10480,Andrew Allen,Consumer


## 4. Create Product Dimension

In [7]:
dim_product = (
    df[
        [
            "product_id",
            "product_name",
            "category",
            "sub_category"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_product.head()

,product_id,product_name,category,sub_category
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels
3,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables
4,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,Office Supplies,Storage


## 5. Create Location Dimension

In [8]:
dim_location = (
    df[
        [
            "country",
            "region",
            "state",
            "city",
            "postal_code"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_location["location_id"] = (
    "LOC" +
    (dim_location.index+1).astype(str).str.zfill(5)
)

dim_location = dim_location[
    [
        "location_id",
        "country",
        "region",
        "state",
        "city",
        "postal_code"
    ]
]

dim_location.head()

,location_id,country,region,state,city,postal_code
0,LOC00001,United States,South,Kentucky,Henderson,42420
1,LOC00002,United States,West,California,Los Angeles,90036
2,LOC00003,United States,South,Florida,Fort Lauderdale,33311
3,LOC00004,United States,West,California,Los Angeles,90032
4,LOC00005,United States,South,North Carolina,Concord,28027


## 6. Create Date Dimension

In [9]:
date_values = (
    df["order_date"]
    .drop_duplicates()
    .sort_values()
)

dim_date = pd.DataFrame({
    "full_date": date_values
})

dim_date["date_id"] = (
    dim_date["full_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = "Q" + dim_date["full_date"].dt.quarter.astype(str)
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()

dim_date.head()

,full_date,date_id,year,quarter,month,month_name
7980,2014-01-03,20140103,2014,Q1,1,January
739,2014-01-04,20140104,2014,Q1,1,January
1759,2014-01-05,20140105,2014,Q1,1,January
5327,2014-01-06,20140106,2014,Q1,1,January
7660,2014-01-07,20140107,2014,Q1,1,January


## 7. Create Fact Sales Table

Grain:

**One row = one product transaction**


In [10]:
fact_sales = df[
    [
        "row_id",
        "order_id",
        "order_date",
        "customer_id",
        "product_id",
        "ship_mode",
        "sales",
        "quantity",
        "discount",
        "profit"
    ]
].copy()

fact_sales = fact_sales.rename(
    columns={"row_id":"sales_id"}
)

fact_sales = fact_sales.merge(
    dim_date[["date_id","full_date"]],
    left_on="order_date",
    right_on="full_date",
    how="left"
)

location_map = df[
    [
        "country",
        "region",
        "state",
        "city",
        "postal_code"
    ]
].drop_duplicates()

location_map = location_map.merge(
    dim_location,
    on=[
        "country",
        "region",
        "state",
        "city",
        "postal_code"
    ]
)

fact_sales = fact_sales.merge(
    location_map[
        [
            "customer_id"
        ]
    ] if False else dim_location,
    how="cross"
)

# Replace with a correct location mapping from raw transaction rows
fact_sales = df[
    [
        "row_id",
        "order_id",
        "order_date",
        "customer_id",
        "product_id",
        "ship_mode",
        "sales",
        "quantity",
        "discount",
        "profit",
        "country",
        "region",
        "state",
        "city",
        "postal_code"
    ]
].copy()

fact_sales = fact_sales.rename(columns={"row_id":"sales_id"})

fact_sales = fact_sales.merge(
    dim_date[["date_id","full_date"]],
    left_on="order_date",
    right_on="full_date",
    how="left"
)

fact_sales = fact_sales.merge(
    dim_location,
    on=[
        "country",
        "region",
        "state",
        "city",
        "postal_code"
    ],
    how="left"
)

fact_sales = fact_sales[
    [
        "sales_id",
        "order_id",
        "date_id",
        "customer_id",
        "product_id",
        "location_id",
        "ship_mode",
        "sales",
        "quantity",
        "discount",
        "profit"
    ]
]

fact_sales.head()


,sales_id,order_id,date_id,customer_id,product_id,location_id,ship_mode,sales,quantity,discount,profit
0,1,CA-2016-152156,20161108,CG-12520,FUR-BO-10001798,LOC00001,Second Class,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,20161108,CG-12520,FUR-CH-10000454,LOC00001,Second Class,731.9400,3,0.00,219.5820
2,3,CA-2016-138688,20160612,DV-13045,OFF-LA-10000240,LOC00002,Second Class,14.6200,2,0.00,6.8714
3,4,US-2015-108966,20151011,SO-20335,FUR-TA-10000577,LOC00003,Standard Class,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,20151011,SO-20335,OFF-ST-10000760,LOC00003,Standard Class,22.3680,2,0.20,2.5164


## 8. Validation

In [11]:
print("Customers:", len(dim_customer))
print("Products:", len(dim_product))
print("Locations:", len(dim_location))
print("Dates:", len(dim_date))
print("Sales:", len(fact_sales))

print("\nMissing keys")
print(fact_sales.isnull().sum())

Customers: 793
Products: 1894
Locations: 632
Dates: 1237
Sales: 9994

Missing keys
sales_id       0
order_id       0
date_id        0
customer_id    0
product_id     0
location_id    0
ship_mode      0
sales          0
quantity       0
discount       0
profit         0
dtype: int64


## 9. Export Tables

In [12]:
output_path = Path(
    r"D:\Retail sales projects\Project project\processed_tables"
)

output_path.mkdir(exist_ok=True)

dim_customer.to_csv(output_path/"dim_customer.csv", index=False)
dim_product.to_csv(output_path/"dim_product.csv", index=False)
dim_location.to_csv(output_path/"dim_location.csv", index=False)
dim_date.to_csv(output_path/"dim_date.csv", index=False)
fact_sales.to_csv(output_path/"fact_sales.csv", index=False)

print("Export completed")

Export completed
